In [9]:
import pandas as pd
import numpy as np
import os
import time
from pathlib import Path

from nba_api.stats.endpoints import leaguedashteamstats
from nba_api.stats.endpoints import leaguedashplayerstats
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.endpoints import leaguedashplayerstats



# Makes nba_teams.csv

In [3]:
from nba_api.stats.static import teams
import pandas as pd

#Get all NBA teams
nba_teams = pd.DataFrame(teams.get_teams())

#Select and rename columns
nba_teams = nba_teams[
    ["id", "abbreviation", "full_name"]
].copy()

nba_teams.columns = [
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "TEAM_NAME"
]

#Save dataset
nba_teams.to_csv("data/nba_teams.csv", index=False)

display(nba_teams)

,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME
0,1610612737,ATL,Atlanta Hawks
1,1610612738,BOS,Boston Celtics
2,1610612739,CLE,Cleveland Cavaliers
3,1610612740,NOP,New Orleans Pelicans
4,1610612741,CHI,Chicago Bulls
5,1610612742,DAL,Dallas Mavericks
6,1610612743,DEN,Denver Nuggets
7,1610612744,GSW,Golden State Warriors
8,1610612745,HOU,Houston Rockets
9,1610612746,LAC,Los Angeles Clippers


# Makes nba_champions csv

In [ ]:
#NBA Champions: 1996-97 through 2025-26

champions = [
    ("1996-97", "CHI"),
    ("1997-98", "CHI"),
    ("1998-99", "SAS"),
    ("1999-00", "LAL"),
    ("2000-01", "LAL"),
    ("2001-02", "LAL"),
    ("2002-03", "SAS"),
    ("2003-04", "DET"),
    ("2004-05", "SAS"),
    ("2005-06", "MIA"),
    ("2006-07", "SAS"),
    ("2007-08", "BOS"),
    ("2008-09", "LAL"),
    ("2009-10", "LAL"),
    ("2010-11", "DAL"),
    ("2011-12", "MIA"),
    ("2012-13", "MIA"),
    ("2013-14", "SAS"),
    ("2014-15", "GSW"),
    ("2015-16", "CLE"),
    ("2016-17", "GSW"),
    ("2017-18", "GSW"),
    ("2018-19", "TOR"),
    ("2019-20", "LAL"),
    ("2020-21", "MIL"),
    ("2021-22", "GSW"),
    ("2022-23", "DEN"),
    ("2023-24", "BOS"),
    ("2024-25", "OKC"),
    ("2025-26", "NYK")
]

#Create DataFrame
championship_history = pd.DataFrame(
    champions,
    columns=["SEASON", "TEAM_ABBREVIATION"]
)

#Add official NBA team IDs and names
championship_history = championship_history.merge(
    nba_teams,
    on="TEAM_ABBREVIATION",
    how="left",
    validate="many_to_one"
)

#Arrange columns
championship_history = championship_history[
    [
        "SEASON",
        "TEAM_ID",
        "TEAM_ABBREVIATION",
        "TEAM_NAME"
    ]
]

#Verify data
print("Total championships:", len(championship_history))
print("Missing values:")
print(championship_history.isnull().sum())

display(championship_history)

championship_history.to_csv("data/nba_champions.csv", index=False)


Total championships: 30
Missing values:
SEASON               0
TEAM_ID              0
TEAM_ABBREVIATION    0
TEAM_NAME            0
dtype: int64


,SEASON,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME
0,1996-97,1610612741,CHI,Chicago Bulls
1,1997-98,1610612741,CHI,Chicago Bulls
2,1998-99,1610612759,SAS,San Antonio Spurs
3,1999-00,1610612747,LAL,Los Angeles Lakers
4,2000-01,1610612747,LAL,Los Angeles Lakers
5,2001-02,1610612747,LAL,Los Angeles Lakers
6,2002-03,1610612759,SAS,San Antonio Spurs
7,2003-04,1610612765,DET,Detroit Pistons
8,2004-05,1610612759,SAS,San Antonio Spurs
9,2005-06,1610612748,MIA,Miami Heat


# Makes team_regular_season_stats_clean csv

In [ ]:
from nba_api.stats.endpoints import leaguedashteamstats
import pandas as pd
import time

seasons = [
    f"{year}-{str(year + 1)[-2:]}"
    for year in range(2010, 2026)
]

def download_team_category(seasons, measure_type):
    all_data = []

    for season in seasons:
        print(f"Downloading {measure_type}: {season}")

        for attempt in range(3):
            try:
                endpoint = leaguedashteamstats.LeagueDashTeamStats(
                    season=season,
                    season_type_all_star="Regular Season",
                    measure_type_detailed_defense=measure_type,
                    per_mode_detailed="PerGame",
                    timeout=60
                )

                df = endpoint.get_data_frames()[0]
                df["SEASON"] = season

                all_data.append(df)

                print(f"Success: {season}")
                time.sleep(2)
                break

            except Exception as e:
                print(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(5)

        else:
            print(f"FAILED: {season}")

    if not all_data:
        raise RuntimeError(f"No {measure_type} data downloaded")

    return pd.concat(all_data, ignore_index=True)


team_base = download_team_category(
    seasons, "Base"
)

team_advanced = download_team_category(
    seasons, "Advanced"
)

team_opponent = download_team_category(
    seasons, "Opponent"
)


In [ ]:
#Merge base and advanced statistics
team_season_stats = team_base.merge(
    team_advanced,
    on=["TEAM_ID", "SEASON"],
    how="outer",
    suffixes=("", "_ADV")
)

#Merge opponent statistics
team_season_stats = team_season_stats.merge(
    team_opponent,
    on=["TEAM_ID", "SEASON"],
    how="left",
    suffixes=("", "_OPP")
)

#Add team abbreviations
team_season_stats = team_season_stats.merge(
    nba_teams[["TEAM_ID", "TEAM_ABBREVIATION"]],
    on="TEAM_ID",
    how="left"
)


,TEAM_ID,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,FGA,FG_PCT,...,OPP_AST_RANK,OPP_TOV_RANK,OPP_STL_RANK,OPP_BLK_RANK,OPP_BLKA_RANK,OPP_PF_RANK,OPP_PFD_RANK,OPP_PTS_RANK,PLUS_MINUS_RANK_OPP,TEAM_ABBREVIATION
0,1610612737,Atlanta Hawks,82,44,38,0.537,48.1,36.2,78.4,0.462,...,16,29,3,5,28,2,30,9,16,ATL
1,1610612737,Atlanta Hawks,66,40,26,0.606,49.0,36.8,81.0,0.454,...,14,9,7,10,23,11,25,6,6,ATL
2,1610612737,Atlanta Hawks,82,44,38,0.537,48.4,37.6,81.0,0.464,...,15,9,25,6,23,6,28,13,13,ATL
3,1610612737,Atlanta Hawks,82,38,44,0.463,48.4,37.3,81.6,0.458,...,24,10,26,12,26,11,24,15,17,ATL
4,1610612737,Atlanta Hawks,82,60,22,0.732,48.1,38.1,81.7,0.466,...,24,6,8,18,15,11,30,5,4,ATL


## Clean that jawn

In [ ]:
#Make a cleaned copy
team_season_stats_clean = team_season_stats.copy()

#Remove duplicate metadata columns
duplicate_columns = [
    "TEAM_NAME_ADV",
    "TEAM_NAME_OPP",
    "GP_ADV", "W_ADV", "L_ADV", "W_PCT_ADV", "MIN_ADV",
    "GP_OPP", "W_OPP", "L_OPP", "W_PCT_OPP", "MIN_OPP"
]

team_season_stats_clean = team_season_stats_clean.drop(
    columns=duplicate_columns,
    errors="ignore"
)

#Remove ranking columns
rank_columns = [
    col for col in team_season_stats_clean.columns
    if "RANK" in col
]

team_season_stats_clean = team_season_stats_clean.drop(
    columns=rank_columns
)

#Move identifying columns to the front
first_columns = [
    "SEASON",
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "TEAM_NAME"
]

remaining_columns = [
    col for col in team_season_stats_clean.columns
    if col not in first_columns
]

team_season_stats_clean = team_season_stats_clean[
    first_columns + remaining_columns
]

#Preview
display(team_season_stats_clean.head())

#Save cleaned dataset
team_season_stats_clean.to_csv(
    "data/team_regular_season_stats_clean.csv",
    index=False
)

print("Rows:", len(team_season_stats_clean))
print("Columns:", len(team_season_stats_clean.columns))
print("Seasons:", team_season_stats_clean["SEASON"].nunique())
print("Teams per season:")
print(team_season_stats_clean.groupby("SEASON")["TEAM_ID"].nunique())

,SEASON,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GP,W,L,W_PCT,MIN,FGM,...,OPP_REB,OPP_AST,OPP_TOV,OPP_STL,OPP_BLK,OPP_BLKA,OPP_PF,OPP_PFD,OPP_PTS,PLUS_MINUS_OPP
0,2010-11,1610612737,ATL,Atlanta Hawks,82,44,38,0.537,48.1,36.2,...,40.6,21.4,12.5,6.7,4.2,4.2,18.5,19.0,95.8,0.8
1,2011-12,1610612737,ATL,Atlanta Hawks,66,40,26,0.606,49.0,36.8,...,42.3,20.5,15.1,7.5,4.7,4.6,19.1,17.8,93.2,-3.4
2,2012-13,1610612737,ATL,Atlanta Hawks,82,44,38,0.537,48.4,37.6,...,43.7,22.3,15.2,8.4,4.3,4.5,18.8,18.0,97.5,-0.4
3,2013-14,1610612737,ATL,Atlanta Hawks,82,38,44,0.463,48.4,37.3,...,43.5,23.2,15.2,8.2,4.3,4.0,20.0,19.2,101.5,0.5
4,2014-15,1610612737,ATL,Atlanta Hawks,82,60,22,0.732,48.1,38.1,...,43.6,23.5,16.1,7.4,4.9,4.6,19.6,17.8,97.1,-5.4


# Makes nba_rosters csv

In [5]:
#Seasons: 2010-11 through 2025-26
seasons = [
    f"{year}-{str(year + 1)[-2:]}"
    for year in range(2010, 2026)
]

all_rosters = []

#Loop through every season
for season in seasons:

    print(f"Downloading {season}...")

    #Loop through every NBA team
    for _, team in nba_teams.iterrows():

        team_id = team["TEAM_ID"]

        try:
            roster = commonteamroster.CommonTeamRoster(
                team_id=team_id,
                season=season,
                timeout=60
            )

            df = roster.get_data_frames()[0]

            #Add team and season information
            df["SEASON"] = season
            df["TEAM_ID"] = team_id
            df["TEAM_ABBREVIATION"] = team["TEAM_ABBREVIATION"]
            df["TEAM_NAME"] = team["TEAM_NAME"]

            all_rosters.append(df)

            print(
                f"  {team['TEAM_ABBREVIATION']}: "
                f"{len(df)} players"
            )

            time.sleep(0.6)

        except Exception as e:
            print(
                f"  ERROR: {team['TEAM_ABBREVIATION']} "
                f"in {season}: {e}"
            )

#Combine all rosters
nba_rosters = pd.concat(all_rosters, ignore_index=True)

print("\nDownload complete!")
print("Total records:", len(nba_rosters))
print("Total seasons:", nba_rosters["SEASON"].nunique())

  ATL: 15 players
  BOS: 15 players
  CLE: 15 players
  NOP: 15 players
  CHI: 15 players
  DAL: 15 players
  DEN: 14 players
  GSW: 14 players
  HOU: 15 players
  LAC: 14 players
  LAL: 15 players
  MIA: 15 players
  MIL: 14 players
  MIN: 13 players
  BKN: 15 players
  NYK: 15 players
  ORL: 13 players
  IND: 15 players
  PHI: 15 players
  PHX: 14 players
  POR: 15 players
  SAC: 13 players
  SAS: 15 players
  OKC: 15 players
  TOR: 15 players
  UTA: 14 players
  MEM: 14 players
  WAS: 15 players
  DET: 15 players
  CHA: 15 players
  ATL: 15 players
  BOS: 15 players
  CLE: 15 players
  NOP: 15 players
  CHI: 14 players
  DAL: 15 players
  DEN: 15 players
  GSW: 15 players
  HOU: 15 players
  LAC: 15 players
  LAL: 14 players
  MIA: 15 players
  MIL: 15 players
  MIN: 15 players
  BKN: 15 players
  NYK: 15 players
  ORL: 15 players
  IND: 15 players
  PHI: 14 players
  PHX: 13 players
  POR: 15 players
  SAC: 14 players
  SAS: 15 players
  OKC: 15 players
  TOR: 14 players
  UTA: 15 

## Clean that jawn

In [ ]:
#Keep useful columns
roster_columns = [
    "SEASON",
    "TEAM_ID",
    "TEAM_ABBREVIATION",
    "TEAM_NAME",
    "PLAYER_ID",
    "PLAYER",
    "NICKNAME",
    "PLAYER_SLUG",
    "NUM",
    "POSITION",
    "HEIGHT",
    "WEIGHT",
    "BIRTH_DATE",
    "AGE",
    "EXP",
    "SCHOOL",
    "HOW_ACQUIRED"
]

nba_rosters = nba_rosters[roster_columns].copy()

#Remove duplicate player-team-season records
nba_rosters = nba_rosters.drop_duplicates(
    subset=["SEASON", "TEAM_ID", "PLAYER_ID"]
)

#Save CSV
nba_rosters.to_csv(
    "data/nba_rosters.csv",
    index=False
)



## Some checks

In [8]:
print("Rows:", len(nba_rosters))
print("Columns:", len(nba_rosters.columns))
print("Seasons:", nba_rosters["SEASON"].nunique())

print("\nPlayers per season:")
print(nba_rosters.groupby("SEASON")["PLAYER_ID"].nunique())

print("\nTeams per season:")
print(nba_rosters.groupby("SEASON")["TEAM_ID"].nunique())

print("\nMissing values:")
print(nba_rosters.isnull().sum())

#Check for duplicate player-team-season records
print(
    "Duplicate records:",
    nba_rosters.duplicated(
        subset=["SEASON", "TEAM_ID", "PLAYER_ID"]
    ).sum()
)

#Check number of players per team-season
team_roster_sizes = (
    nba_rosters.groupby(["SEASON", "TEAM_ABBREVIATION"])
    ["PLAYER_ID"]
    .nunique()
)

print("\nSmallest team roster:")
print(team_roster_sizes.idxmin(), team_roster_sizes.min())

print("\nLargest team roster:")
print(team_roster_sizes.idxmax(), team_roster_sizes.max())

# Check positions
print("\nPlayer positions:")
print(nba_rosters["POSITION"].value_counts(dropna=False))

# Check experience values
print("\nNBA experience values:")
print(nba_rosters["EXP"].value_counts().sort_index())

Rows: 7747
Columns: 17
Seasons: 16

Players per season:
SEASON
2010-11    437
2011-12    442
2012-13    446
2013-14    448
2014-15    450
2015-16    450
2016-17    451
2017-18    502
2018-19    497
2019-20    510
2020-21    503
2021-22    506
2022-23    508
2023-24    532
2024-25    534
2025-26    530
Name: PLAYER_ID, dtype: int64

Teams per season:
SEASON
2010-11    30
2011-12    30
2012-13    30
2013-14    30
2014-15    30
2015-16    30
2016-17    30
2017-18    30
2018-19    30
2019-20    30
2020-21    30
2021-22    30
2022-23    30
2023-24    30
2024-25    30
2025-26    30
Name: TEAM_ID, dtype: int64

Missing values:
SEASON                  0
TEAM_ID                 0
TEAM_ABBREVIATION       0
TEAM_NAME               0
PLAYER_ID               0
PLAYER                  0
NICKNAME                0
PLAYER_SLUG             0
NUM                    33
POSITION                0
HEIGHT                  0
WEIGHT                  6
BIRTH_DATE              0
AGE                     0
EXP     

# Makes player_season_stats.csv

In [12]:
#Seasons from 2010-11 through 2025-26
seasons = [
    f"{year}-{str(year + 1)[-2:]}"
    for year in range(2010, 2026)
]

all_player_stats = []

for season in seasons:
    print(f"Downloading {season}...")

    try:
        player_stats = leaguedashplayerstats.LeagueDashPlayerStats(
            season=season,
            season_type_all_star="Regular Season",
            per_mode_detailed="PerGame",
            measure_type_detailed_defense="Base",
            timeout=60
        )

        df = player_stats.get_data_frames()[0]
        df["SEASON"] = season

        all_player_stats.append(df)

        print(f"  Downloaded {len(df)} player-team records")

        time.sleep(0.6)

    except Exception as e:
        print(f"  ERROR downloading {season}: {e}")

#Combine all seasons
nba_player_stats = pd.concat(all_player_stats, ignore_index=True)

print("\nDownload finished!")
print("Total records:", len(nba_player_stats))


  Downloaded 452 player-team records
  Downloaded 478 player-team records
  Downloaded 469 player-team records
  Downloaded 482 player-team records
  Downloaded 492 player-team records
  Downloaded 476 player-team records
  Downloaded 486 player-team records
  Downloaded 540 player-team records
  Downloaded 530 player-team records
  Downloaded 529 player-team records
  Downloaded 540 player-team records
  Downloaded 605 player-team records
  Downloaded 539 player-team records
  Downloaded 572 player-team records
  Downloaded 569 player-team records
  Downloaded 582 player-team records

Download finished!
Total records: 8341


## Clean that jawn


In [ ]:
#Remove ranking columns
rank_columns = [
    col for col in nba_player_stats.columns
    if col.endswith("_RANK")
]

nba_player_stats = nba_player_stats.drop(columns=rank_columns)

#Remove fantasy-related columns
fantasy_columns = [
    "NBA_FANTASY_PTS",
    "WNBA_FANTASY_PTS",
    "FP_HIGH_SCORE"
]

nba_player_stats = nba_player_stats.drop(
    columns=fantasy_columns,
    errors="ignore"
)

#Move SEASON to the first column
columns = ["SEASON"] + [
    col for col in nba_player_stats.columns
    if col != "SEASON"
]

nba_player_stats = nba_player_stats[columns]

#Save CSV
nba_player_stats.to_csv(
    "data/player_regular_season_stats.csv",
    index=False
)

print("Dataset cleaned and saved!")
print("Rows:", len(nba_player_stats))
print("Columns:", len(nba_player_stats.columns))


Dataset cleaned and saved!
Rows: 8341
Columns: 36


## Some checks

In [14]:
print("Records per season:")
print(nba_player_stats.groupby("SEASON").size())

print("\nUnique players per season:")
print(
    nba_player_stats.groupby("SEASON")["PLAYER_ID"].nunique()
)

print("\nTeams per season:")
print(
    nba_player_stats.groupby("SEASON")["TEAM_ID"].nunique()
)

print("\nMissing values:")
print(
    nba_player_stats.isna().sum().sort_values(
        ascending=False
    ).head(10)
)

print("\nSample rows:")
display(nba_player_stats.head())


Records per season:
SEASON
2010-11    452
2011-12    478
2012-13    469
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
2025-26    582
dtype: int64

Unique players per season:
SEASON
2010-11    452
2011-12    478
2012-13    469
2013-14    482
2014-15    492
2015-16    476
2016-17    486
2017-18    540
2018-19    530
2019-20    529
2020-21    540
2021-22    605
2022-23    539
2023-24    572
2024-25    569
2025-26    582
Name: PLAYER_ID, dtype: int64

Teams per season:
SEASON
2010-11    30
2011-12    30
2012-13    30
2013-14    30
2014-15    30
2015-16    30
2016-17    30
2017-18    30
2018-19    30
2019-20    30
2020-21    30
2021-22    30
2022-23    30
2023-24    30
2024-25    30
2025-26    30
Name: TEAM_ID, dtype: int64

Missing values:
SEASON               0
PLAYER_ID            0
PLAYER_NAME          0
NICKNAME             0
TEAM_ID              0
TEAM_

,SEASON,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,AGE,GP,W,L,...,STL,BLK,BLKA,PF,PFD,PTS,PLUS_MINUS,DD2,TD3,TEAM_COUNT
0,2010-11,201985,AJ Price,AJ,1610612754,IND,24.0,50,22,28,...,0.6,0.0,0.2,1.2,1.5,6.5,0.5,0,0,1
1,2010-11,201166,Aaron Brooks,Aaron,1610612756,PHX,26.0,59,26,33,...,0.6,0.1,0.7,1.9,2.1,10.7,-2.8,2,0,2
2,2010-11,201189,Aaron Gray,Aaron,1610612740,NOH,26.0,41,21,20,...,0.3,0.3,0.3,2.3,0.7,3.1,0.7,2,0,1
3,2010-11,201151,Acie Law,Acie,1610612744,GSW,26.0,51,20,31,...,0.6,0.0,0.4,1.2,1.2,4.2,-2.6,0,0,2
4,2010-11,1733,Al Harrington,Al,1610612743,DEN,31.0,73,45,28,...,0.5,0.1,0.6,2.8,1.5,10.5,1.8,1,0,1
